# Module 10 — Modules, packages and the project

Nine modules of one file at a time. This is the one about the second file — and
about what turns a folder of them into something you can install, run by name, and
hand to somebody else.

`sensorlib/` next to this notebook is a real package. The cells below import from it.

## 1. Importing runs the file

There is no separate declaration step. `import x` finds `x`, **executes it top to
bottom**, and binds the resulting module object to the name.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "sensorlib").is_dir() else Path.cwd() / "10_modules"
sys.path.insert(0, str(HERE))  # section 2 explains this line

import sensorlib.limits  # noqa: E402 -- the path has to be set before the import

print("first import done")

`sensorlib/limits.py` has a `print` at the top, and you have just seen it. Now import
it again.

(`ruff` reports the line below as `F811`, "redefinition of unused name". It is right
about the code and it is also the demonstration: a second import contributes nothing.
The `# noqa` keeps it.)

In [ ]:
import sensorlib.limits  # noqa: F811 -- ruff calls this a redefinition, which is the point

print("second import done -- and nothing printed above this line")
print("sensorlib.limits" in sys.modules)

The second import does not execute anything. Python keeps every module it has
imported in `sys.modules`, keyed by name, and an import that finds the name there
hands the existing object straight back.

Two consequences worth having:

- **A module's top-level code is setup that runs once.** Opening a file, connecting to
  something, or printing at import time happens whenever somebody imports you, in an
  order you do not control. Keep it to definitions and constants.
- **Editing a module does not affect an already-running program.** In a notebook, a
  changed file needs a kernel restart — which is why the "I fixed it and it still
  fails" moment always happens here first.

## 2. `sys.path`, and why the same code works in one place and not another

`import x` searches a plain list of directories, in order, and takes the first match.

In [ ]:
import sys

for entry in sys.path[:5]:
    print(repr(entry))

Java's `classpath` is the same idea, with two differences: it is passed to the tool
rather than being a mutable list inside the program, and a Java file *declares* the
package it belongs to. Python has no such declaration — **where a file sits on
`sys.path` is the whole of what makes it importable**, which is why an import error is
almost always about this list rather than about your code.

The first entry is the interesting one. It is not the same thing depending on how you
started the program:

| how you start it | `sys.path[0]` |
| --- | --- |
| `python tool/show.py` | the **file's** folder — `tool/` |
| `python -m tool.show` | the **current** directory |

That single difference is behind most of the import errors you will meet. A script
that imports a sibling package works with `-m` from the project root and fails when
run by its path, because in the second case the project root is not on the list at
all.

`sys.path.insert(0, ...)` in the cell above is the blunt version of fixing that.
Section 7 is the version you should actually ship.

## 3. `if __name__ == "__main__":`

Because importing a file runs it, a file that both defines things and does things
needs to tell the two situations apart. `__name__` is how.

In [ ]:
import sensorlib.parsing

print(sensorlib.parsing.__name__)  # when imported: its dotted name
print(__name__)  # in a notebook or a script: __main__

So the idiom reads: *if this file is the one that was started, rather than one that
was imported, do the following.*

```python
def main():
    ...


if __name__ == "__main__":
    main()
```

Without it, importing your script to reuse one function would also run whatever it
does at the bottom. Java's `public static void main` is a declaration the launcher
looks for; this is an ordinary `if` on an ordinary string, evaluated while the file
runs.

`python -m x` sets `__name__` to `"__main__"` as well, which is why both ways of
starting a program reach the same `main()`.

## 4. A package is a folder

`sensorlib/` is a package because it is a folder Python can find on `sys.path`.
`__init__.py` is what runs when the package is first imported, and it is where a
package decides what it offers directly.

In [ ]:
import sensorlib

print(sensorlib.__name__, sensorlib.VERSION)
print(sensorlib.to_reading("21.7"))  # re-exported in __init__.py from sensorlib.parsing
print(sensorlib.__all__)
print(
    Path(sensorlib.__file__).name
)  # the package IS its __init__.py, as far as import is concerned

`__init__.py` has been optional since Python 3.3 — a folder without one still imports,
as a *namespace package*. Write one anyway: it is the difference between a package
that states what it is and a folder that happens to import.

`__all__` is a list of names, and it does one job: it is what `from sensorlib import
*` hands out. Without it, the star form takes every name not starting with an
underscore — including modules you imported for your own use, which then appear in
the caller's namespace as if you had offered them.

Which is one of two reasons not to write `import *` at all. The other is that a reader
of the calling file cannot tell where a name came from.

In [ ]:
# sensorlib/__init__.py imports from sensorlib.parsing. Given that, and given that an
# import caches, how many of these are the same object?
import sensorlib
import sensorlib.parsing
from sensorlib.parsing import to_reading

assert (sensorlib.to_reading is to_reading) == ...
assert (sensorlib.parsing.to_reading is to_reading) == ...

Importing a name gives you the object, not a copy of it. `from x import y` and `x.y`
reach the same object — the difference is only whether you keep the module name in
your own file. That matters when the module rebinds `y` later: your `from`-imported
name still points at the old object.

## 5. Circular imports

Two modules that import each other are a design problem before they are an error, and
Python's error message says so — it names the module that is *partially initialised*.

In [ ]:
# a.py:  from b import thing
# b.py:  from a import other
#
# ImportError: cannot import name 'thing' from partially initialized module 'b'
#              (most likely due to a circular import)
#
# The first import starts running b, which starts running a, which asks b for a name
# that b has not got to yet. There is no way round it at import time.
print("read the comment above -- the message names the cause")

The fixes, in the order to try them:

1. **Move the shared thing into a third module** that both import. Usually the two
   modules were one idea split in the wrong place.
2. **Import inside the function** rather than at the top, so the import happens after
   both modules exist. Legal, occasionally right, and a smell when it is the only fix
   you can find.
3. **Import the module, not the name** — `import b` then `b.thing` at call time —
   which works because the module object exists from the start and only its contents
   arrive late.

## 6. The standard library, and two names owed since module 06

`import` reaches the standard library the same way it reaches your own code: those
modules are simply on `sys.path`. Module 06 counted with `dict.get` and grouped with
`setdefault`, and promised the shorter versions here.

In [ ]:
import collections

log = [("TH-04", 91.0), ("TH-01", 21.7), ("TH-04", 88.0)]

print(collections.Counter(tag for tag, _ in log))
print(collections.Counter(tag for tag, _ in log).most_common(1))

grouped = collections.defaultdict(list)  # the argument is a function that makes the default
for tag, value in log:
    grouped[tag].append(value)

print(dict(grouped))

`defaultdict` has one behaviour worth knowing before you reach for it: **reading a
missing key creates it.** That is how the `append` above works without a `setdefault`,
and it means a lookup can change the dict.

In [ ]:
import collections

grouped = collections.defaultdict(list)
print("before:", dict(grouped))

if grouped["TH-99"]:  # just asking -- or so it looks
    print("has readings")

print("after: ", dict(grouped))  # the key exists now

So `defaultdict` where you are building, and a plain `dict` where you are asking. It
is a `dict` in every other respect — `isinstance(grouped, dict)` is `True`, and
`dict(grouped)` gives you an ordinary one to hand on.

## 7. A project

`sys.path.insert` at the top of a file is what you do when there is no project. With
one, the import works because the package is installed into the environment — and
`uv` does that for you.

```console
$ uv init --app --name sensortool sensortool
$ tree sensortool
sensortool/
├── .python-version
├── README.md
├── pyproject.toml
└── src
    └── sensortool
        └── __init__.py
```

Four things to notice.

**`src/`.** The package sits one level down, so the project root is *not* on
`sys.path` when you run tests. That means your tests import the package the way a user
would — installed — rather than accidentally finding the source folder. It is a small
layout decision that removes a whole class of "works until it is installed".

**`pyproject.toml` is the project.** Name, version, `requires-python`, dependencies,
and the build backend:

```toml
[project]
name = "sensortool"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = []

[project.scripts]
sensortool = "sensortool:main"
```

**`[project.scripts]` turns a function into a command.** `name = "module:function"`,
and afterwards `uv run sensortool` runs it. That is how a tool gets a name instead of
a path.

**The lock file.** `uv add requests` writes the dependency into `pyproject.toml` and
the exact resolved versions into `uv.lock`. `pyproject.toml` says what you want,
`uv.lock` says what you got, and `uv sync` reproduces the second one exactly — on your
machine, on a colleague's, in CI. Commit both.

The commands, in the order you meet them:

| command | what it does |
| --- | --- |
| `uv init --app --name x x` | new project: `pyproject.toml`, `src/x/`, `.python-version` |
| `uv add requests` | dependency into `pyproject.toml`, exact version into `uv.lock` |
| `uv sync` | make the environment match the lock file |
| `uv run x` | run the command from `[project.scripts]` |
| `uv run python -m x` | run the package's `__main__.py` |
| `uv run pytest` | run the tests in that environment |

`uv run` syncs first, so there is no separate "activate the environment" step and no
way to run against the wrong one — which is the actual reason this course uses `uv`
throughout rather than `pip` and a `venv`.

## 8. Arguments

`sys.argv` is the raw list: the program name first, then what the shell handed over.
For anything past one argument, `argparse` is in the standard library and does the
parsing, the types, the `--help` and the error message.

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="report", description="Summarise a sensor log.")
parser.add_argument("path")  # positional, required
parser.add_argument("--limit", type=float, default=85.0)  # converted for you
parser.add_argument("--verbose", action="store_true")  # a flag: present or not

args = parser.parse_args(["data/readings.csv", "--limit", "90"])
print(args)
print(args.limit, type(args.limit).__name__)

parser.print_help()

`parse_args()` with no argument reads `sys.argv`; passing a list is how you test it.
On a bad value it prints the usage line and **exits the process** — `SystemExit`, not
an exception you catch. That is right for a command-line tool and worth knowing before
it surprises you inside a test.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

That is the end of Part 2. Part 3 is objects: classes, `@dataclass`, and the protocols
that make your own types work with `for`, `in` and `with`.